<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/demo_SIMC_til_heftet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# =============================================================================
# 1. INTERAKTIVE KOMPONENTER
# =============================================================================
y0_input = widgets.FloatText(value=20.0, description='y0 (Start):')
ysp_input = widgets.FloatText(value=65.0, description='y_sp (Mål):')
K_slider = widgets.FloatSlider(value=13.01, min=1.0, max=30.0, step=0.01, description='K (Prosessf.):')
T_slider = widgets.IntSlider(value=130, min=10, max=600, step=10, description='T (Tidsk.):')
L_slider = widgets.IntSlider(value=4, min=0, max=50, step=1, description='L (Dødtid):')

lambda_dropdown = widgets.Dropdown(
    options=[1, 2, 4, 6, 8, 10],
    value=2,
    description='\u03BB-faktor:'
)

metning_checkbox = widgets.Checkbox(value=False, description='Aktiver pådragsmetning (0-100%)')

kontroll_panel = widgets.VBox([
    widgets.Label(value="NIVÅINNSTILLINGER"), y0_input, ysp_input,
    widgets.HTML(value="<br>"),
    widgets.Label(value="PROSESSENS EGENSKAPER"), K_slider, T_slider, L_slider,
    widgets.HTML(value="<br>"),
    widgets.Label(value="REGULATOR HASTIGHET / MODUS"), lambda_dropdown, metning_checkbox
], layout=widgets.Layout(margin='40px 0px 0px 30px'))

plot_utdata = widgets.Output()

# =============================================================================
# 2. SIMULERING OG DYNAMISK PLOTTING
# =============================================================================
def simuler_og_plott(change=None):
    y0 = y0_input.value
    y_sp = ysp_input.value
    K = K_slider.value
    T = T_slider.value
    L = L_slider.value
    lambda_faktor = lambda_dropdown.value
    bruk_metning = metning_checkbox.value

    u0 = y0 / K
    lambda_val = T / lambda_faktor

    if lambda_val < L:
        lambda_val = L

    Kp = (1 / K) * (T / (lambda_val + L))
    Ti = min(T, 4 * (lambda_val + L))

    dt = 0.5
    t_slutt = 5*T
    t_vektor = np.arange(0, t_slutt, dt)
    N = len(t_vektor)

    settpunkt = np.ones(N) * y0
    y_aapen = np.ones(N) * y0
    y_lukket = np.ones(N) * y0
    u_lukket = np.ones(N) * u0
    u_aapen_sprangverdi = u0 + (y_sp - y0) / K

    dødtid_steps = int(round(L / dt))
    u_historikk_lukket = [u0] * (dødtid_steps + 1)
    u_historikk_aapen = [u0] * (dødtid_steps + 1)
    integratør = 0.0

    sprang_tid = 30.0

    for k in range(0, N - 1):
        if t_vektor[k] >= sprang_tid:
            settpunkt[k+1] = y_sp
            u_aapen_naa = u_aapen_sprangverdi
        else:
            settpunkt[k+1] = y0
            u_aapen_naa = u0

        # --- LUKKET SLØYFE ---
        avvik = settpunkt[k] - y_lukket[k]
        P_ledd = Kp * avvik
        integratør += (Kp / Ti) * avvik * dt

        u_temp = u0 + P_ledd + integratør
        if bruk_metning:
            u_lukket[k] = np.clip(u_temp, 0.0, 100.0)
        else:
            u_lukket[k] = u_temp

        u_historikk_lukket.append(u_lukket[k])
        u_forsinket_lukket = u_historikk_lukket[- (dødtid_steps + 1)]
        derivert_y_lukket = (-(y_lukket[k] - y0) + K * (u_forsinket_lukket - u0)) / T
        y_lukket[k+1] = y_lukket[k] + derivert_y_lukket * dt

        # --- ÅPEN SLØYFE ---
        u_historikk_aapen.append(u_aapen_naa)
        u_forsinket_aapen = u_historikk_aapen[- (dødtid_steps + 1)]
        derivert_y_aapen = (-(y_aapen[k] - y0) + K * (u_forsinket_aapen - u0)) / T
        y_aapen[k+1] = y_aapen[k] + derivert_y_aapen * dt

    u_lukket[N-1] = u_lukket[N-2]

    sprang_amplitude = y_sp - y0
    y_632 = y0 + 0.632 * sprang_amplitude
    y_982 = y0 + 0.982 * sprang_amplitude

    # FIKSET: Legger til [0][0] for å hente ut kun det FØRSTE tidspunktet i listen
    krysninger_lambda = np.where(y_lukket >= y_632)
    t_lambda_faktisk = t_vektor[krysninger_lambda[0][0]] if len(krysninger_lambda[0]) > 0 else None

    krysninger_T = np.where(y_aapen >= y_632)
    t_T_faktisk = t_vektor[krysninger_T[0][0]] if len(krysninger_T[0]) > 0 else None

    krysninger_4lambda = np.where(y_lukket >= y_982)
    t_4lambda_faktisk = t_vektor[krysninger_4lambda[0][0]] if len(krysninger_4lambda[0]) > 0 else None

    krysninger_4T = np.where(y_aapen >= y_982)
    t_4T_faktisk = t_vektor[krysninger_4T[0][0]] if len(krysninger_4T[0]) > 0 else None

    with plot_utdata:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9.5, 8.5))

        # Tegn hovedlinjer og legg inn labels for Legend
        ax1.axhline(y=y_sp, color='red', linestyle='--', linewidth=2.5, label=f'Settpunkt ({y_sp:.1f}%)')
        if lambda_faktor > 1:
            ax1.plot(t_vektor, y_lukket, 'b-', label='Lambda-regulering', linewidth=3.5)
        ax1.plot(t_vektor, y_aapen, 'g-', label='Åpen sløyfe', linewidth=3.5)

        ax1.axhline(y=y_632, color='#555555', linestyle='--', linewidth=1.5, label=f'63.2% av spranget ({y_632:.1f}%)')
        ax1.axhline(y=y_982, color='#888888', linestyle='-.', linewidth=1.5, label=f'98.2% av spranget ({y_982:.1f}%)')

        # Vertikal linje for spranget
        ax1.axvline(x=sprang_tid, color='red', linestyle=':', linewidth=1.5)
        ax1.text(sprang_tid + 3, 93, 'Sprang t=30s', color='red', fontsize=10, fontweight='bold')

        # Vertikale tidskonstant-merknader (Nå som rene singel-verdier uten feilmeldinger!)
        if lambda_faktor > 1 and t_lambda_faktisk is not None and t_lambda_faktisk < t_slutt:
            ax1.axvline(x=t_lambda_faktisk, color='blue', linestyle=':', linewidth=2)
            lambda_avlest = t_lambda_faktisk - sprang_tid - L
            ax1.text(t_lambda_faktisk + 3, y0 + 3, f'\u03BB = {lambda_avlest:.1f}s\n(t = {t_lambda_faktisk:.1f}s)', color='blue', fontsize=10, fontweight='bold')

        if t_T_faktisk is not None and t_T_faktisk < t_slutt:
            ax1.axvline(x=t_T_faktisk, color='green', linestyle=':', linewidth=2)
            T_avlest = t_T_faktisk - sprang_tid - L
            ax1.text(t_T_faktisk + 3, y0 + 35, f'T = {T_avlest:.0f}s\n(t = {t_T_faktisk:.0f}s)', color='green', fontsize=10, fontweight='bold')

        if lambda_faktor > 1 and t_4lambda_faktisk is not None and t_4lambda_faktisk < t_slutt:
            ax1.axvline(x=t_4lambda_faktisk, color='#4ba3e3', linestyle='--', linewidth=1.5)
            tid_4lam_avlest = t_4lambda_faktisk - sprang_tid - L
            ax1.text(t_4lambda_faktisk + 3, y_982 - 24, f'4\u03BB = {tid_4lam_avlest:.1f}s\n(t = {t_4lambda_faktisk:.1f}s)', color='#1d6fa5', fontsize=10, fontweight='bold')

        if t_4T_faktisk is not None and t_4T_faktisk < t_slutt:
            ax1.axvline(x=t_4T_faktisk, color='#5cb85c', linestyle='--', linewidth=1.5)
            tid_4T_avlest = t_4T_faktisk - sprang_tid - L
            ax1.text(t_4T_faktisk + 3, y_982 - 12, f'4T = {tid_4T_avlest:.0f}s\n(t = {t_4T_faktisk:.0f}s)', color='darkgreen', fontsize=10, fontweight='bold')

        # Akseoppsett og den rene Legend-boksen nede til høyre
        ax1.set_ylabel('Nivå [%]', fontsize=14)
        ax1.set_ylim(-5, 105)
        ax1.tick_params(labelsize=12)
        ax1.grid(True)
        ax1.legend(loc='lower right', fontsize=11, framealpha=0.95)

        modus_tekst = "MED pådragsbegrensning" if bruk_metning else "UTEN pådragsbegrensning (Teori)"
        ax1.set_title(f'SIMC: Kp = {Kp:.3f}  |  Ti = {Ti:.1f} s  ({modus_tekst})', fontsize=14, pad=15)

        # Nederste graf (Pådrag)
        ax2.plot(t_vektor, u_lukket, 'black', label='$u_{lukket}$', linewidth=2.5)
        ax2.axhline(u_aapen_sprangverdi, color='g', linestyle=':', label='$u_{open}$', linewidth=2.5)
        ax2.axvline(x=sprang_tid, color='red', linestyle=':', linewidth=1.5)
        ax2.set_xlabel('Tid [sekunder]', fontsize=14)
        ax2.set_ylabel('Pådrag [%]', fontsize=14)

        u_maks = np.max([np.max(u_lukket), u_aapen_sprangverdi, u0])
        u_min_val = np.min([np.min(u_lukket), u_aapen_sprangverdi, u0])
        ax2.set_ylim(u_min_val - 5, u_maks + 5)

        ax2.tick_params(labelsize=12)
        ax2.grid(True)
        ax2.legend(loc='lower right', fontsize=12)

        plt.tight_layout()
        plt.show()

for widget in [y0_input, ysp_input, K_slider, T_slider, L_slider, lambda_dropdown, metning_checkbox]:
    widget.observe(simuler_og_plott, names='value')

app_layout = widgets.HBox([plot_utdata, kontroll_panel])
display(app_layout)

simuler_og_plott()
